# HGSEL Speed Benchmark
Compare original vs optimized HGSEL layer.

In [ ]:
!pip install torch --quiet

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import torch
import argparse
from typing import Dict, List

## Import original and optimized implementations

In [ ]:
# Original implementations
from hgsel.layer.hgsel_layer import HGSELLayer as HGSELLayerOriginal
from hgsel.routing.hash_functions import MultiHashRouter as MultiHashRouterOriginal
from hgsel.expert.expert_bank import ExpertBank as ExpertBankOriginal

# Optimized implementations
from hgsel.layer.hgsel_layer_fast import HGSELLayerFast
from hgsel.routing.hash_functions_fast import MultiHashRouterFast, InvertedDispatchExpertBank

In [ ]:
# Config
BATCH_SIZE = 32
SEQ_LEN = 128
D_MODEL = 512
D_FF = 2048
N_EXPERTS = 64
K_ACTIVE = 2
N_HASHES = 4
N_RUNS = 100
WARMUP = 10

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

In [ ]:
# Create inputs
hidden_states = torch.randn(
    BATCH_SIZE, SEQ_LEN, D_MODEL,
    device=device
)
hidden_states_flat = hidden_states.view(-1, D_MODEL)
print(f"Input shape: {hidden_states.shape} = {hidden_states_flat.shape[0]} tokens")

## Create layers

In [ ]:
# Original
original_router = MultiHashRouterOriginal(
    n_experts=N_EXPERTS, k_active=K_ACTIVE, n_hashes=N_HASHES,
    hidden_dim=D_MODEL
).to(device)

original_expert_bank = ExpertBankOriginal(
    n_experts=N_EXPERTS, k_active=K_ACTIVE, d_model=D_MODEL, d_ff=D_FF
).to(device)

original_layer = HGSELLayerOriginal(
    d_model=D_MODEL, d_ff=D_FF, n_experts=N_EXPERTS,
    k_active=K_ACTIVE, n_hashes=N_HASHES
).to(device)

# Optimized
fast_router = MultiHashRouterFast(
    n_experts=N_EXPERTS, k_active=K_ACTIVE, n_hashes=N_HASHES,
    hidden_dim=D_MODEL
).to(device)

fast_expert_bank = InvertedDispatchExpertBank(
    n_experts=N_EXPERTS, k_active=K_ACTIVE, d_model=D_MODEL, d_ff=D_FF
).to(device)

fast_layer = HGSELLayerFast(
    d_model=D_MODEL, d_ff=D_FF, n_experts=N_EXPERTS,
    k_active=K_ACTIVE, n_hashes=N_HASHES
).to(device)

fast_layer_bf16 = HGSELLayerFast(
    d_model=D_MODEL, d_ff=D_FF, n_experts=N_EXPERTS,
    k_active=K_ACTIVE, n_hashes=N_HASHES, use_bf16=True
).to(device)

print("Layers created!")

## Benchmark function

In [ ]:
def benchmark(name, model, inputs, n_runs=N_RUNS, warmup=WARMUP):
    """Benchmark a layer."""
    # Warmup
    for _ in range(warmup):
        _ = model(inputs)
    
    if device == 'cuda':
        torch.cuda.synchronize()
    
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = model(inputs)
        if device == 'cuda':
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)
    
    return {
        'name': name,
        'mean_ms': sum(times) / len(times),
        'min_ms': min(times),
        'max_ms': max(times),
        'std_ms': (sum((t - sum(times)/len(times))**2 for t in times) / len(times)) ** 0.5,
    }

## Run benchmarks

In [ ]:
print("\n=== ROUTING BENCHMARK ===")
r_orig = benchmark("Original", original_router, hidden_states_flat)
r_fast = benchmark("Fast", fast_router, hidden_states_flat)
print(f"Original: {r_orig['mean_ms']:.3f} ms")
print(f"Fast:     {r_fast['mean_ms']:.3f} ms")
print(f"Speedup:  {r_orig['mean_ms']/r_fast['mean_ms']:.2f}x\n")

In [ ]:
print("=== FULL LAYER BENCHMARK ===\n")
results = []

for name, layer in [
    ("Original (fp32)", original_layer),
    ("Fast (fp32)", fast_layer),
    ("Fast (bf16)", fast_layer_bf16),
]:
    print(f"Running {name}...", end=" ", flush=True)
    r = benchmark(name, layer, hidden_states)
    results.append(r)
    print(f"{r['mean_ms']:.3f} ms")

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df['speedup'] = df['mean_ms'].iloc[0] / df['mean_ms']
df.columns = ['Name', 'Mean (ms)', 'Min', 'Max', 'Std', 'Speedup vs Original']
df

## Experiment: n_hashes = k_active

In [ ]:
print("=== n_hashes = k_active ===\n")

# Test H=2 (same as k_active) vs H=4
router_h2_orig = MultiHashRouterOriginal(
    n_experts=N_EXPERTS, k_active=K_ACTIVE, n_hashes=2,
    hidden_dim=D_MODEL
).to(device)

router_h2_fast = MultiHashRouterFast(
    n_experts=N_EXPERTS, k_active=K_ACTIVE, n_hashes=2,
    hidden_dim=D_MODEL
).to(device)

r_h4_orig = benchmark("Original H=4", original_router, hidden_states_flat)
r_h2_orig = benchmark("Original H=2", router_h2_orig, hidden_states_flat)
r_h2_fast = benchmark("Fast H=2", router_h2_fast, hidden_states_flat)

print(f"Original (H=4): {r_h4_orig['mean_ms']:.3f} ms")
print(f"Original (H=2): {r_h2_orig['mean_ms']:.3f} ms")
print(f"Fast (H=2):     {r_h2_fast['mean_ms']:.3f} ms")
print(f"\nSpeedup (H=2 fast vs H=4 original): {r_h4_orig['mean_ms']/r_h2_fast['mean_ms']:.2f}x")

## Summary

In [ ]:
print("\n" + "="*50)
print("BENCHMARK COMPLETE")
print("="*50)
print(f"\nConfig: batch={BATCH_SIZE}, seq={SEQ_LEN}, d_model={D_MODEL}")
print(f"Experts: N={N_EXPERTS}, k={K_ACTIVE}, H={N_HASHES}")
print(f"Device: {device}\n")